In [ ]:
pip install pymysql sqlalchemy

In [2]:
import pandas as pd,numpy as np

In [3]:
# Importing the csv File
df=pd.read_csv('customer_shopping_behavior.csv')

In [4]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [17]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_usd', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases', 'age_group'],
      dtype='str')

In [6]:
df.shape

(3900, 18)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [8]:
# converted to snake casing
df.columns=(df.columns.str.replace(" ","_")).str.lower()

In [9]:
# removed the ( )
df=df.rename(columns={'purchase_amount_(usd)':'purchase_amount_usd'})

In [ ]:
# created a new colum for age group
if_else_conditions = [
    (df["age"] >= 30) & (df["age"] < 40),  # 30 to 39
    (df["age"] >= 40) & (df["age"] < 60),  # 40 to 59
    (df["age"] >= 60),  # 60 and above
]
if_else_labels = [ "Adults", "Middle-age", "Senior"]

df["age_group"] = np.select(
    if_else_conditions, if_else_labels, default='Young'
)

In [ ]:
# craeted a new colum that showed the purchse frequency dates
freq_map={
    'Fortnightly':14,
    'Weekly':7,
    'Annually':365,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Monthly': 30,
    'Every 3 Months':90
}
df['purchases_frequency']=df['frequency_of_purchases'].map(freq_map)

In [ ]:
df['frequency_of_purchases'].unique()

<StringArray>
[   'Fortnightly',         'Weekly',       'Annually',      'Quarterly',
      'Bi-Weekly',        'Monthly', 'Every 3 Months']
Length: 7, dtype: str

In [ ]:
# drop_duplicates(): If Customer #1 bought 5 items, their age and gender appear 5 times in your CSV.
# This command deletes those copies, leaving exactly one unique row per customer.

dim_customers = df[['customer_id', 'age', 'gender', 'age_group', 'location']].drop_duplicates().reset_index(drop=True)

In [ ]:
# The Connector: It keeps customer_id inside it. This acts as a map pointer (Foreign Key) so the database can
# link the transaction back to the customer profile without repeating their age/gender every time.


fact_transactions=df[['customer_id', 'item_purchased', 'category', 'purchase_amount_usd', 
                        'size', 'color', 'season', 'review_rating', 'subscription_status', 
                        'shipping_type', 'discount_applied', 'promo_code_used', 
                        'previous_purchases', 'payment_method', 'frequency_of_purchases']]

In [20]:
dim_customers.shape

(3900, 5)

In [27]:
import pymysql

# 1. Connect to MySQL without specifying a database yet
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='aroot', # Replace with your actual MySQL password
    port=3306
)

# 2. Command MySQL to create the database container
cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS shopping_analytics;")
conn.close()

print("Database container 'shopping_analytics' successfully built!")


Database container 'shopping_analytics' successfully built!


In [29]:
from sqlalchemy import create_engine

USER = 'root'            
PASSWORD = 'aroot' # Replace with your actual MySQL password
HOST = 'localhost'        
PORT = '3306'            
DATABASE = 'shopping_analytics' 

connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(connection_string)

# Push your dataframes directly into your new database tables
dim_customers.to_sql(name='dim_customers', con=engine, if_exists='replace', index=False)
fact_transactions.to_sql(name='fact_transactions', con=engine, if_exists='replace', index=False)

print("Successfully exported both tables to MySQL!")


Successfully exported both tables to MySQL!
